# 07 Change Detection — prototype + metrics

**Author:** Serhii Polishchuk  
**Config:** `configs/change_run.yaml`  

Pipeline: ORB align → absdiff → opening → connected components → events JSON  
Pairs: motion + static + synthetic illumination (2–4).

In [ ]:
!pip install -q opencv-python-headless pyyaml matplotlib pandas
import cv2, numpy as np, json, random, shutil, zipfile, urllib.request
from pathlib import Path
import yaml, pandas as pd
import matplotlib.pyplot as plt

OUT = Path("/content/change_out")
OUT.mkdir(parents=True, exist_ok=True)
for s in ["diff", "mask", "overlay", "events"]:
    (OUT / s).mkdir(exist_ok=True)

CFG = {
    "seed": 42,
    "align": {"max_features": 2000, "good_match_ratio": 0.75, "ransac_reproj_threshold": 3.0, "min_inliers": 25},
    "diff": {"tau": 25, "blur_ksize": 5, "opening_ksize": 3, "thr_area": 80},
    "events": {"min_area": 80, "max_events_per_pair": 50},
    "eval": {"tau_grid": [10, 15, 20, 25, 30, 40, 50]},
}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("OpenCV", cv2.__version__)

## 1. Data: ordered frames + pairs

Prefer a short video-like strip. If VisDrone MOT is heavy, we build an ordered strip from publicly downloadable images + synthetic pairs.

In [ ]:
DATA = Path("/content/change_data")
DATA.mkdir(exist_ok=True)
SEQ = DATA / "seq"
SEQ.mkdir(exist_ok=True)

# Download a few sequential-looking aerial frames (Ultralytics bus/zidane not aerial).
# Use picsum seeded stills as base + shifts for motion proxy, OR local DET if present.
det = Path("/content/datasets/VisDrone/VisDrone2019-DET-val/images")
frames = sorted(det.glob("*.jpg"))[:40] if det.exists() else []

if len(frames) < 20:
    # Create synthetic sequence: one base image downloaded, then shifted/crops for motion pairs
    base_path = DATA / "base.jpg"
    if not base_path.exists():
        urllib.request.urlretrieve(
            "https://ultralytics.com/images/bus.jpg", str(base_path)
        )
    base = cv2.imread(str(base_path))
    h, w = base.shape[:2]
    frames = []
    for i in range(24):
        # small translation to simulate camera/object motion
        tx, ty = int(3 * i), int(1.5 * i)
        M = np.float32([[1, 0, tx], [0, 1, ty]])
        warped = cv2.warpAffine(base, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        # add a moving blob on later frames
        img = warped.copy()
        if i >= 8:
            x0 = 50 + 8 * (i - 8)
            cv2.rectangle(img, (x0, 80), (x0 + 60, 140), (0, 0, 255), -1)
        fp = SEQ / f"f_{i:03d}.jpg"
        cv2.imwrite(str(fp), img)
        frames.append(fp)
else:
    for i, f in enumerate(frames):
        dst = SEQ / f"f_{i:03d}.jpg"
        if not dst.exists():
            shutil.copy(f, dst)
        frames[i] = dst

frames = sorted(SEQ.glob("f_*.jpg"))
print("n frames", len(frames))

# Build pairs: static (near), motion (far), synth light
pairs = []
# static: small gap
for a, b, pid in [(0, 1, "static_0_1"), (5, 6, "static_5_6"), (10, 11, "static_10_11"), (15, 16, "static_15_16")]:
    pairs.append({"id": pid, "t0": frames[a], "t1": frames[b], "kind": "static", "expect_change": False, "frame": b, "t": float(b)})
# motion: larger gap / object moved
for a, b, pid in [(0, 8, "motion_0_8"), (4, 12, "motion_4_12"), (8, 16, "motion_8_16"), (10, 20, "motion_10_20")]:
    pairs.append({"id": pid, "t0": frames[a], "t1": frames[b], "kind": "motion", "expect_change": True, "frame": b, "t": float(b)})

# synthetic illumination on same frame
synth_src = frames[3]
img = cv2.imread(str(synth_src))
for name, fn in [
    ("synth_bright_p40", lambda x: cv2.convertScaleAbs(x, alpha=1.0, beta=40)),
    ("synth_bright_m40", lambda x: cv2.convertScaleAbs(x, alpha=1.0, beta=-40)),
    ("synth_gain_1p2", lambda x: cv2.convertScaleAbs(x, alpha=1.2, beta=0)),
]:
    p0 = SEQ / f"{name}_0.jpg"
    p1 = SEQ / f"{name}_1.jpg"
    cv2.imwrite(str(p0), img)
    cv2.imwrite(str(p1), fn(img))
    pairs.append({"id": name, "t0": p0, "t1": p1, "kind": "synth_light", "expect_change": False, "frame": 3, "t": 3.0})

print("n pairs", len(pairs), [p["id"] for p in pairs])

# write change_run.yaml into out
yaml_txt = """seed: 42
align:
  max_features: 2000
  good_match_ratio: 0.75
  ransac_reproj_threshold: 3.0
  min_inliers: 25
diff:
  tau: 50
  blur_ksize: 5
  opening_ksize: 3
  thr_area: 80
events:
  min_area: 80
  max_events_per_pair: 50
eval:
  tau_grid: [10, 15, 20, 25, 30, 40, 50]
pairs:
"""
for p in pairs:
    yaml_txt += f"  - id: {p['id']}\n    kind: {p['kind']}\n    expect_change: {p['expect_change']}\n"
(OUT / "change_run.yaml").write_text(yaml_txt)
print("config saved", OUT / "change_run.yaml")

## 2. Align / diff / events

In [ ]:
def align_t0_to_t1(img0, img1, cfg=None):
    cfg = cfg or CFG["align"]
    g0 = cv2.cvtColor(img0, cv2.COLOR_BGR2GRAY)
    g1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(cfg["max_features"])
    k0, d0 = orb.detectAndCompute(g0, None)
    k1, d1 = orb.detectAndCompute(g1, None)
    if d0 is None or d1 is None or len(k0) < 8:
        return img0.copy(), np.eye(3), 0, True
    bf = cv2.BFMatcher(cv2.NORM_HAMMING)
    knn = bf.knnMatch(d0, d1, k=2)
    good = []
    for m_n in knn:
        if len(m_n) < 2:
            continue
        m, n = m_n
        if m.distance < cfg["good_match_ratio"] * n.distance:
            good.append(m)
    if len(good) < 8:
        return img0.copy(), np.eye(3), len(good), True
    src = np.float32([k0[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, cfg["ransac_reproj_threshold"])
    inliers = int(mask.sum()) if mask is not None else 0
    if H is None or inliers < cfg["min_inliers"]:
        return img0.copy(), np.eye(3), inliers, True
    h, w = img1.shape[:2]
    warped = cv2.warpPerspective(img0, H, (w, h))
    return warped, H, inliers, False

def frame_diff_mask(warped0, img1, tau, cfg=None):
    cfg = cfg or CFG["diff"]
    k = cfg["blur_ksize"] | 1
    a = cv2.GaussianBlur(cv2.cvtColor(warped0, cv2.COLOR_BGR2GRAY), (k, k), 0)
    b = cv2.GaussianBlur(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), (k, k), 0)
    diff = cv2.absdiff(a, b)
    _, bw = cv2.threshold(diff, int(tau), 255, cv2.THRESH_BINARY)
    ok = cfg["opening_ksize"] | 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ok, ok))
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)
    residual = float(diff.mean())
    return diff, bw, residual

def build_change_events(mask, pair_meta, tau, min_area=None):
    min_area = min_area or CFG["events"]["min_area"]
    n, labels, stats, cents = cv2.connectedComponentsWithStats(mask, connectivity=8)
    events = []
    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        x, y, w, h = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP], stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        # change_score: normalized mean diff inside component (proxy)
        events.append({
            "frame": int(pair_meta["frame"]),
            "t": float(pair_meta["t"]),
            "change_score": round(min(1.0, area / 5000.0), 4),
            "area": area,
            "xyxy": [int(x), int(y), int(x + w), int(y + h)],
            "centroid": [float(cents[i][0]), float(cents[i][1])],
            "track_id": None,
            "pair_id": pair_meta["id"],
            "component_id": i,
        })
        if len(events) >= CFG["events"]["max_events_per_pair"]:
            break
    return events

print("helpers ok")

In [ ]:
TAU_WORK = 50  # will confirm after curve
summary = []
all_events = {}

for p in pairs:
    img0 = cv2.imread(str(p["t0"]))
    img1 = cv2.imread(str(p["t1"]))
    warped, H, inliers, rejected = align_t0_to_t1(img0, img1)
    diff, mask, residual = frame_diff_mask(warped, img1, TAU_WORK)
    events = build_change_events(mask, p, TAU_WORK)
    all_events[p["id"]] = events

    cv2.imwrite(str(OUT / "diff" / f"{p['id']}.png"), diff)
    cv2.imwrite(str(OUT / "mask" / f"{p['id']}.png"), mask)
    # overlay
    ov = img1.copy()
    ov[mask > 0] = (0.4 * ov[mask > 0] + 0.6 * np.array([0, 0, 255])).astype(np.uint8)
    for e in events:
        x1, y1, x2, y2 = e["xyxy"]
        cv2.rectangle(ov, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imwrite(str(OUT / "overlay" / f"{p['id']}.png"), ov)

    (OUT / "events" / f"{p['id']}.json").write_text(json.dumps({
        "pair_id": p["id"], "tau": TAU_WORK, "inliers": inliers,
        "rejected_align": rejected, "residual_mean": residual,
        "events": events
    }, indent=2))

    summary.append({
        "pair_id": p["id"], "kind": p["kind"], "expect_change": p["expect_change"],
        "inliers": inliers, "rejected_align": rejected, "residual_mean": round(residual, 2),
        "n_events": len(events)
    })
    print(p["id"], "inliers", inliers, "rej", rejected, "events", len(events))

pd.DataFrame(summary).to_csv(OUT / "pairs_summary.csv", index=False)
print("saved pairs_summary.csv")

## 3. P/R/F1 curve over τ

Proxy labels: motion → must have ≥1 event (TP/FN); static/synth → any event = FP.

In [ ]:
curve = []
for tau in CFG["eval"]["tau_grid"]:
    TP = FP = FN = 0
    for p in pairs:
        img0 = cv2.imread(str(p["t0"]))
        img1 = cv2.imread(str(p["t1"]))
        warped, H, inliers, rejected = align_t0_to_t1(img0, img1)
        diff, mask, residual = frame_diff_mask(warped, img1, tau)
        events = build_change_events(mask, p, tau)
        has = len(events) > 0
        if p["expect_change"]:
            if has: TP += 1
            else: FN += 1
        else:
            if has: FP += 1
    prec = TP / (TP + FP + 1e-9)
    rec = TP / (TP + FN + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    curve.append({"tau": tau, "TP": TP, "FP": FP, "FN": FN,
                  "precision": round(prec, 3), "recall": round(rec, 3), "F1": round(f1, 3)})
    print(curve[-1])

df = pd.DataFrame(curve)
df.to_csv(OUT / "pr_f1_curve.csv", index=False)
best = df.loc[df["F1"].idxmax()]
print("working tau* =", int(best["tau"]), "F1", best["F1"])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["tau"], df["precision"], "-o", label="P")
ax.plot(df["tau"], df["recall"], "-o", label="R")
ax.plot(df["tau"], df["F1"], "-o", label="F1")
ax.axvline(best["tau"], color="k", ls="--", alpha=0.5)
ax.set_xlabel("tau"); ax.legend(); ax.set_title("P/R/F1 vs tau")
fig.tight_layout()
fig.savefig(OUT / "pr_f1_curve.png", dpi=120)
plt.show()

## 4. Show FP (synth) vs true change (motion) overlays

In [ ]:
show = ["synth_bright_p40", "synth_gain_1p2", "motion_0_8", "static_0_1"]
fig, axes = plt.subplots(1, len(show), figsize=(3 * len(show), 3))
for ax, sid in zip(axes, show):
    path = OUT / "overlay" / f"{sid}.png"
    im = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(im); ax.set_title(sid, fontsize=9); ax.axis("off")
fig.tight_layout()
fig.savefig(OUT / "fp_vs_true_panel.png", dpi=120)
plt.show()
print("panel saved")

## 5. Event format example + README

```bash
# One-shot: Run all cells in this notebook
# Outputs: /content/change_out/{events,diff,mask,overlay,pr_f1_curve.csv,pairs_summary.csv,change_run.yaml}
```

In [ ]:
# pack one example event file with required fields
ex_id = "motion_0_8"
ex = json.loads((OUT / "events" / f"{ex_id}.json").read_text())
print(json.dumps(ex, indent=2)[:800])
print("OUT tree:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(OUT))